# 24 — Multi-Class Attack Classification (in-domain + cross-dataset)

**Dijalankan di SageMaker.** Menambah (bukan mengganti) eksperimen biner: di sini
model memprediksi *kategori* serangan, bukan sekadar attack/benign. Menjawab
reviewer A dengan eksperimen multi-kelas sungguhan (confusion matrix, macro-F1,
recall per-kelas, MCC multi-kelas).

Dua evaluasi jujur:
1. **In-domain per dataset**: CIC (Benign + DDoS/DoS/BruteForce/Botnet/Infiltration)
   dan UNSW (Benign + Generic/Exploits/Fuzzers/DoS/Recon/...). XGBoost `multi:softprob`.
2. **Cross-dataset pada kategori sepadan**: latih di satu dataset, uji di dataset
   lain, hanya untuk kelas grup umum yang ada di KEDUA dataset (Benign, DoS, DDoS/flood).

Sumber data sama seperti notebook 23: CIC `file_100.csv` (9 fitur CICFlowMeter +
`Label`), UNSW `UNSW_NB15_training-set.csv` (9 fitur + `attack_cat`).

Output: `multiclass_results.json` + confusion PNG + CSV per-kelas -> S3 `unsw-far/multiclass/`.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('xgboost','scikit-learn','scipy','pandas','numpy','matplotlib','boto3') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
    matthews_corrcoef, f1_score, balanced_accuracy_score)
from sklearn.preprocessing import StandardScaler, LabelEncoder
import xgboost as xgb
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='multiclass_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; MIN_CLASS=200  # kelas dgn < MIN_CLASS sampel digabung ke 'Other-rare' agar stabil
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Pemetaan label -> kategori (sama seperti notebook 23) + muat data

In [ ]:
def map_cic(lbl):
    s=str(lbl).strip().lower()
    if s in ('benign','normal'): return 'Benign'
    if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
    if s.startswith('dos'): return 'DoS'
    if 'bruteforce' in s or 'brute force' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
    if s=='bot' or 'botnet' in s: return 'Botnet'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'Web'
    return 'Other'
def map_uns(lbl):
    s=str(lbl).strip().lower()
    if s in ('normal','benign',''): return 'Benign'
    return {'dos':'DoS','exploits':'Exploits','fuzzers':'Fuzzers','generic':'Generic',
            'reconnaissance':'Recon','backdoor':'Backdoor','backdoors':'Backdoor',
            'shellcode':'Shellcode','worms':'Worms','analysis':'Analysis'}.get(s,'Other')

def load_cic():
    p=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
    if not p: print('CIC csv tak ada'); return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    if not all(v in c.columns for v in cm.values()): print('CIC kolom kurang'); return None
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON})
    d['cat']=c[LAB].map(map_cic)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

def load_uns():
    p=first(['../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])
    if not p: print('UNSW csv tak ada'); return None
    u2=pd.read_csv(p); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','attack_cat']
    if not all(x in u2.columns for x in need): print('UNSW kolom kurang'); return None
    d=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                    'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                    'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
    d['cat']=u2['attack_cat'].fillna('Normal').map(map_uns)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

cic=load_cic(); uns=load_uns()
for nm,df in [('CIC',cic),('UNSW',uns)]:
    if df is not None: print(nm, df['cat'].value_counts().to_dict())
print('=== SEL 2 (muat + peta) SELESAI ===')

## 3. Fungsi latih+uji multi-kelas in-domain (XGBoost multi:softprob)

In [ ]:
def merge_rare(df, min_n=MIN_CLASS):
    vc=df['cat'].value_counts(); rare=[c for c,n in vc.items() if n<min_n and c!='Benign']
    if rare:
        df=df.copy(); df['cat']=df['cat'].where(~df['cat'].isin(rare),'Other-rare')
    return df

def run_indomain(df, name):
    if df is None or len(df)==0: return None
    df=merge_rare(df)
    X=df[CANON].values; le=LabelEncoder(); y=le.fit_transform(df['cat'].values)
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,random_state=SEED,stratify=y)
    sc=StandardScaler().fit(Xtr)  # FIT hanya pada train (no leakage)
    Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)
    clf=xgb.XGBClassifier(objective='multi:softprob',num_class=len(le.classes_),
        max_depth=8,learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=-1)
    clf.fit(Xtr,ytr)
    yp=clf.predict(Xte)
    labels=list(le.classes_)
    cm=confusion_matrix(yte,yp)
    rep=classification_report(yte,yp,target_names=labels,output_dict=True,zero_division=0)
    res={'dataset':name,'classes':labels,'n_test':int(len(yte)),
         'macro_f1':round(float(f1_score(yte,yp,average='macro')),4),
         'weighted_f1':round(float(f1_score(yte,yp,average='weighted')),4),
         'mcc':round(float(matthews_corrcoef(yte,yp)),4),
         'balanced_acc':round(float(balanced_accuracy_score(yte,yp)),4),
         'per_class':{c:{'precision':round(rep[c]['precision'],4),'recall':round(rep[c]['recall'],4),
                         'f1':round(rep[c]['f1-score'],4),'support':int(rep[c]['support'])} for c in labels},
         'confusion':cm.tolist()}
    print(f"[{name}] macro-F1={res['macro_f1']} MCC={res['mcc']} bal-acc={res['balanced_acc']}")
    return res, cm, labels

def plot_cm(cm, labels, title, fname, normalize=True):
    M=np.array(cm,float)
    if normalize:
        row=M.sum(1,keepdims=True); row[row==0]=1; M=M/row
    fig,ax=plt.subplots(figsize=(1.3+0.7*len(labels),1.1+0.7*len(labels)))
    im=ax.imshow(M,cmap='Blues',vmin=0,vmax=1 if normalize else None)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j,i,f'{M[i,j]:.2f}' if normalize else int(M[i,j]),ha='center',va='center',
                    fontsize=7,color='white' if M[i,j]>0.5 else 'black')
    ax.set_title(title); fig.colorbar(im,ax=ax,fraction=0.046)
    plt.tight_layout(); savefig(fname)
print('=== SEL 3 (fungsi multi-kelas) SELESAI ===')

## 4. Jalankan in-domain multi-kelas: CIC & UNSW

In [ ]:
RESULTS['in_domain']={}
for name,df in [('CIC',cic),('UNSW',uns)]:
    out=run_indomain(df,name)
    if out is None: continue
    res,cm,labels=out
    RESULTS['in_domain'][name]=res
    plot_cm(cm,labels,f'{name} multi-class (row-normalized recall)',f'confusion_{name}.png',normalize=True)
    # tabel per-kelas
    print(f'\n=== {name} per-class ===')
    import IPython.display as ipd
    ipd.display(pd.DataFrame(res['per_class']).T[['precision','recall','f1','support']])
print('=== SEL 4 (in-domain CIC & UNSW) SELESAI ===')

## 5. Cross-dataset pada kategori sepadan (grup umum yang ada di KEDUA dataset)

Hanya kelas yang muncul di CIC dan UNSW dapat diuji-silang multi-kelas. Grup umum
yang beririsan: **Benign, DoS**. (DDoS hanya CIC; Exploits/Generic hanya UNSW.)
Kami latih di sumber pada kelas irisan, uji di target pada kelas irisan yang sama.

In [ ]:
def run_cross(src,src_name,tgt,tgt_name,shared):
    if src is None or tgt is None: return None
    s=src[src['cat'].isin(shared)].copy(); t=tgt[tgt['cat'].isin(shared)].copy()
    if s['cat'].nunique()<2 or t['cat'].nunique()<2: print('kelas irisan <2, lewati'); return None
    le=LabelEncoder().fit(shared)
    Xs=StandardScaler().fit(s[CANON].values)  # fit scaler pada SUMBER train
    Xs_tr=Xs.transform(s[CANON].values); ys=le.transform(s['cat'].values)
    Xt_te=Xs.transform(t[CANON].values); yt=le.transform(t['cat'].values)  # target pakai scaler sumber (deploy realistis)
    clf=xgb.XGBClassifier(objective='multi:softprob',num_class=len(shared),max_depth=8,learning_rate=0.1,
        n_estimators=200,subsample=0.8,colsample_bytree=0.8,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=-1)
    clf.fit(Xs_tr,ys); yp=clf.predict(Xt_te)
    cm=confusion_matrix(yt,yp,labels=range(len(shared)))
    res={'direction':f'{src_name}->{tgt_name}','classes':shared,'n_test':int(len(yt)),
         'macro_f1':round(float(f1_score(yt,yp,average='macro',zero_division=0)),4),
         'mcc':round(float(matthews_corrcoef(yt,yp)),4) if len(set(yt))>1 else None,
         'confusion':cm.tolist()}
    print(f"[{res['direction']}] classes={shared} macro-F1={res['macro_f1']} MCC={res['mcc']}")
    return res,cm,shared

RESULTS['cross_dataset']=[]
SHARED=['Benign','DoS']
for a,an,b,bn in [(cic,'CIC',uns,'UNSW'),(uns,'UNSW',cic,'CIC')]:
    out=run_cross(a,an,b,bn,SHARED)
    if out is None: continue
    res,cm,labels=out; RESULTS['cross_dataset'].append(res)
    plot_cm(cm,labels,f"{res['direction']} (shared classes, recall)",f"confusion_cross_{an}_to_{bn}.png",normalize=True)
print('=== SEL 5 (cross-dataset kelas sepadan) SELESAI ===')

## 6. Ringkas + simpan + UPLOAD S3

In [ ]:
# Ringkasan in-domain untuk tabel paper
rows=[]
for nm,r in RESULTS.get('in_domain',{}).items():
    rows.append({'dataset':nm,'n_classes':len(r['classes']),'macro_F1':r['macro_f1'],
                 'weighted_F1':r['weighted_f1'],'MCC':r['mcc'],'balanced_acc':r['balanced_acc']})
summ=pd.DataFrame(rows)
print('Ringkasan in-domain multi-kelas:'); 
import IPython.display as ipd; ipd.display(summ)
summ.to_csv(os.path.join(OUTDIR,'multiclass_summary.csv'),index=False)
jp=os.path.join(OUTDIR,'multiclass_results.json')
json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/multiclass/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/multiclass/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 6 (simpan + upload) SELESAI ===')
print('SEMUA SELESAI. Beri tahu asisten -> unduh s3://%s/%s/multiclass/ untuk masuk paper + notebook 20.'%(S3_BUCKET,S3_PREFIX))